# MAGeCK processing workflow for pooled bulk CRISPR screens

This notebook provides a **generic example** of the MAGeCK workflow used to process pooled bulk CRISPR-screen data for FITdb. 

The workflow supports two common starting points:

1. **FASTQ files** → `mageck count` → sgRNA count table
2. **An existing sgRNA count matrix** → formatting/control definition → `mageck test`

An optional final section shows how `mageck pathway` can be run on MAGeCK gene-level results.


### Expected input formats

**sgRNA library file for `mageck count`**

MAGeCK accepts a library annotation containing sgRNA IDs, sequences, and target genes. See the MAGeCK documentation for the exact format required by the installed version.

**Count matrix for `mageck test`**

A tab-delimited matrix with the first two columns corresponding to sgRNA and gene identifiers, followed by one column per sample/replicate, for example:

| sgRNA | Gene | input_rep1 | input_rep2 | treatment_rep1 | treatment_rep2 |
|---|---|---:|---:|---:|---:|
| sgRNA_001 | GENE1 | 120 | 135 | 30 | 28 |
| sgRNA_002 | GENE1 | 95 | 102 | 18 | 22 |

In [ ]:
from pathlib import Path
import itertools
import subprocess

import pandas as pd

## 1. Optional: generate sgRNA counts directly from FASTQ files

Skip this section if an sgRNA count matrix has already been generated.

Replace the example paths and sample names below with the files for a given screen. The example assumes that each selected population is compared with the same input/control library. Running `mageck count` separately for each comparison is convenient for documentation; alternatively, all FASTQs can be counted together in a single MAGeCK command.

In [ ]:
# Example project-relative paths
PROJECT_DIR = Path("/path/to/crispr_screen")
FASTQ_DIR = PROJECT_DIR / "fastq"
LIBRARY_FILE = PROJECT_DIR / "library" / "sgRNA_library.csv"
COUNT_OUTPUT_DIR = PROJECT_DIR / "mageck_count"
COUNT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared input/control sample
CONTROL_FASTQ = "input.fastq.gz"
CONTROL_LABEL = "input"

# Example experimental populations
EXPERIMENTAL_FASTQS = [
    "sample_A.fastq.gz",
    "sample_B.fastq.gz",
    "sample_C.fastq.gz",
]

for fastq_name in EXPERIMENTAL_FASTQS:
    # Remove FASTQ suffixes to create a readable sample label
    sample_label = fastq_name
    for suffix in [".fastq.gz", ".fq.gz", ".fastq", ".fq"]:
        if sample_label.endswith(suffix):
            sample_label = sample_label[:-len(suffix)]
            break

    output_prefix = COUNT_OUTPUT_DIR / sample_label

    command = [
        "mageck", "count",
        "-l", str(LIBRARY_FILE),
        "-n", str(output_prefix),
        "--sample-label", f"{sample_label},{CONTROL_LABEL}",
        "--fastq",
        str(FASTQ_DIR / fastq_name),
        str(FASTQ_DIR / CONTROL_FASTQ),
    ]

    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

print("MAGeCK counting completed.")

### Alternative: count all samples in one command

For screens containing many samples, a single `mageck count` command can be used with all FASTQ files and a matching comma-separated list of sample labels. The resulting `*.count.txt` file can then be used directly by `mageck test`.

## 2. Prepare an existing sgRNA count matrix

Some published screens provide sgRNA-level counts directly rather than raw FASTQ files. In that case, we first select and modify the columns to match the source dataset.

In [ ]:
PROJECT_DIR = Path("/path/to/crispr_screen")
RAW_DIR = PROJECT_DIR / "raw"
MAGECK_DIR = PROJECT_DIR / "mageck_test"
MAGECK_DIR.mkdir(parents=True, exist_ok=True)

# Example input spreadsheet supplied with a published screen
raw_count_file = RAW_DIR / "sgRNA_counts.xlsx"

counts = pd.read_excel(raw_count_file)

# Rename source-specific columns into a consistent format when needed.
# The first two columns used by MAGeCK should identify the sgRNA and target gene.
counts = counts.rename(columns={
    "guide_id": "sgRNA",
    "target_gene": "Gene",
})

sample_columns = [
    "input_rep1", "input_rep2", "input_rep3",
    "treatment_rep1", "treatment_rep2", "treatment_rep3",
]

counts = counts[["sgRNA", "Gene"] + sample_columns].copy()

# Optional: harmonize labels used for non-targeting control guides.
# Modify this rule according to the annotation used by the source library.
def standardize_control_gene(gene_name):
    gene_name = str(gene_name)
    if "control" in gene_name.lower() or "non-targeting" in gene_name.lower():
        return "NonTargetingControl"
    return gene_name

counts["Gene"] = counts["Gene"].apply(standardize_control_gene)

formatted_count_file = RAW_DIR / "counts_for_mageck.txt"
counts.to_csv(formatted_count_file, sep="\t", index=False)

print(f"Saved MAGeCK-formatted count matrix: {formatted_count_file}")

## 3. Create a non-targeting control sgRNA list

When non-targeting control guides are available, they will be supplied to MAGeCK and used for normalization with `--norm-method control`.

In [ ]:
control_sgrnas = counts.loc[
    counts["Gene"].eq("NonTargetingControl"),
    "sgRNA"
].dropna().drop_duplicates()

control_sgrna_file = RAW_DIR / "non_targeting_sgRNAs.txt"
control_sgrnas.to_csv(control_sgrna_file, index=False, header=False)

print(f"Number of non-targeting control sgRNAs: {len(control_sgrnas)}")
print(f"Saved control sgRNA list: {control_sgrna_file}")

## 4. Run MAGeCK test

Replicates are grouped by biological condition. For each requested comparison, MAGeCK is run with the treatment replicates specified by `-t` and control/input replicates specified by `-c`.

This example uses non-targeting sgRNAs for normalization:

- `--control-sgrna <file>` specifies the control guide list.
- `--norm-method control` estimates the normalization factor from those guides.

MAGeCK produces both sgRNA-level and gene-level summary files. 

In [ ]:
count_file = RAW_DIR / "counts_for_mageck.txt"
control_sgrna_file = RAW_DIR / "non_targeting_sgRNAs.txt"

# Define biological conditions and their replicate columns.
conditions = {
    "Input": ["input_rep1", "input_rep2", "input_rep3"],
    "Treatment": ["treatment_rep1", "treatment_rep2", "treatment_rep3"],
}

# Explicit comparisons are preferable to automatically testing every pair,
# because the biologically appropriate reference can differ among screens.
comparisons = [
    ("Treatment", "Input"),  # (treatment, control)
]

for treatment, control in comparisons:
    treatment_reps = ",".join(conditions[treatment])
    control_reps = ",".join(conditions[control])
    output_prefix = MAGECK_DIR / f"{treatment}_vs_{control}"

    command = [
        "mageck", "test",
        "-k", str(count_file),
        "-t", treatment_reps,
        "-c", control_reps,
        "--control-sgrna", str(control_sgrna_file),
        "--norm-method", "control",
        "-n", str(output_prefix),
    ]

    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

print("MAGeCK test completed.")

## 5. Optional: append screen metadata to MAGeCK output

For database integration, screen-specific metadata can be appended after MAGeCK has finished. This does **not** alter the MAGeCK statistics; it only makes downstream aggregation easier.

In [ ]:
for treatment, control in comparisons:
    output_prefix = MAGECK_DIR / f"{treatment}_vs_{control}"

    for suffix in ["sgrna_summary.txt", "gene_summary.txt"]:
        result_file = Path(f"{output_prefix}.{suffix}")
        if result_file.exists():
            result = pd.read_csv(result_file, sep="\t")
            result["Treatment"] = treatment
            result["Control"] = control
            result.to_csv(result_file, sep="\t", index=False)

## 6. MAGeCK pathway analysis

MAGeCK pathway analysis can be applied to a gene-ranking file together with a GMT gene-set collection. This step depends on the downstream analysis performed for a particular screen.

In [ ]:
GENE_SET_FILE = PROJECT_DIR / "gene_sets" / "example_gene_sets.gmt"
PATHWAY_DIR = PROJECT_DIR / "mageck_pathway"
PATHWAY_DIR.mkdir(parents=True, exist_ok=True)

for gene_summary_file in MAGECK_DIR.glob("*.gene_summary.txt"):
    output_prefix = PATHWAY_DIR / gene_summary_file.name.replace(".gene_summary.txt", "")

    command = [
        "mageck", "pathway",
        "--gene-ranking", str(gene_summary_file),
        "--gmt", str(GENE_SET_FILE),
        "--output-prefix", str(output_prefix),
    ]

    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

print("MAGeCK pathway analysis completed.")

## Notes for adapting this workflow to additional screens

Because FITdb integrates CRISPR screens from multiple publications, the exact raw-data format varies across studies. The screen-specific preprocessing step may therefore require changes to file parsing, sample-column names, or non-targeting-control annotations. After counts are represented as an sgRNA-by-sample matrix, the core MAGeCK testing workflow is consistent.